In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Define symbolic variables
n = sp.symbols('n', integer=True)
A_1, A_2 = sp.symbols('A_1 A_2')

display(Markdown("### Analytical Symbolic Output"))

# Coefficients of the difference equation: alpha_0 = 1, alpha_1 = -1, alpha_2 = -2
alpha = [1.0, -1.0, -2.0]

# Automatically compute the roots of the characteristic polynomial
roots = np.roots([alpha[0], alpha[1], alpha[2]])

display(Markdown(r"**Computed Characteristic Roots ($\lambda$):** $\lambda_1 = " + f"{roots[0]:.3f}" + r"$, $\lambda_2 = " + f"{roots[1]:.3f}$"))

# Setup initial conditions / Vandermonde system for A1, A2 using the computed roots
eq1 = sp.Eq(A_1 + A_2, 1.0 / alpha[0])
eq2 = sp.Eq(A_1 * (roots[0]**(-1)) + A_2 * (roots[1]**(-1)), 0.0)
sol = sp.solve((eq1, eq2), (A_1, A_2))

# Round coefficients to 3 decimal places and convert roots to clean integers/floats inside SymPy
a1_val = round(float(sol[A_1]), 3)
a2_val = round(float(sol[A_2]), 3)
r1_val = round(float(roots[0]), 3)
r2_val = round(float(roots[1]), 3)

# Display constants A1 and A2 rounded to 3 decimal places
display(Markdown(f"**Calculated Constants:** $A_1 = {a1_val}$, $A_2 = {a2_val}$"))

# Intermediate response h_beta[n] (H_V) with clean parentheses around bases
h_beta = (a1_val * (sp.Integer(r1_val) if r1_val.is_integer() else r1_val)**n + 
          a2_val * (sp.Integer(r2_val) if r2_val.is_integer() else r2_val)**n) * sp.Heaviside(n)

# Final impulse response h[n] with clean parentheses around bases
h_final = (a1_val * (sp.Integer(r1_val) if r1_val.is_integer() else r1_val)**(n-1) + 
           a2_val * (sp.Integer(r2_val) if r2_val.is_integer() else r2_val)**(n-1)) * sp.Heaviside(n-1)

display(Markdown(r"**Intermediate Response ($h_{\beta}[n]$):**"))
display(sp.Eq(sp.Symbol('h_\\beta[n]'), h_beta))

display(Markdown(r"**Final Impulse Response $h[n]$:**"))
display(sp.Eq(sp.Symbol('h[n]'), h_final))

# Numerical evaluation for plotting
n_vals = np.arange(0, 16)
h_beta_vals = (a1_val * (roots[0]**n_vals) + a2_val * (roots[1]**n_vals)) * (n_vals >= 0)

h_final_vals = np.zeros_like(n_vals, dtype=float)
for idx, val in enumerate(n_vals):
    if val >= 1:
        n_sub = val - 1
        h_final_vals[idx] = a1_val * (roots[0]**n_sub) + a2_val * (roots[1]**n_sub)

# Plotting the responses
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].stem(n_vals, h_beta_vals, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[0].set_title(r'Intermediate Response $h_{\beta}[n]$', fontsize=9.5, fontweight='bold', color='darkblue')
axes[0].set_ylabel('Amplitude', fontsize=8.5)
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].stem(n_vals, h_final_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
axes[1].set_title(r'Final System Impulse Response $h[n]$', fontsize=9.5, fontweight='bold', color='darkred')
axes[1].set_xlabel('Index $n$', fontsize=8.5)
axes[1].set_ylabel('Amplitude', fontsize=8.5)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()